# 02 — Selection de features et classification

**Cohorte FCCSS** : prediction des pathologies cardiaques severes (grade >= 3 CTCAE) chez les survivants de cancers pediatriques traites par radiotherapie thoracique.

| Donnee | Valeur |
|---|---|
| Patients | 1 375 |
| Variables | 23 |
| Cible | `Pathologie_cardiaque` (binaire) |
| Prevalence | 17.2 % |

**Plan du notebook** :
1. Chargement des donnees et definition des groupes de variables
2. Analyse univariee multi-modeles (LR, RF, GBDT, SVM)
3. Protocole de selection des features (6 etapes)
4. Etude d'ablation — gender
5. Etude d'ablation — variables dosimetriques
6. Modeles multivaries (5-fold CV stratifie)
7. Nested CV feature selection (model-specific)
8. Resume et limites

---
## 1. Chargement des donnees

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from collections import Counter, defaultdict

from scipy.stats import chi2_contingency, mannwhitneyu

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score, recall_score, balanced_accuracy_score,
    make_scorer, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score, f1_score,
)

import os, warnings
warnings.filterwarnings('ignore')

print('Imports OK')

In [ ]:
# ── Publication style ──────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Georgia'],
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
})
sns.set_theme(style='whitegrid')

# Palette projet
COLORS_MODELS = {
    'LogReg': '#1f4e79',
    'RF':     '#2e8b57',
    'GBDT':   '#c17c32',
    'SVM':    '#6a4c93',
}
# Same palette keyed by long name (used in multivariate sections)
MODEL_COLORS_LONG = {
    'Logistic Regression': '#1f4e79',
    'Random Forest':       '#2e8b57',
    'Gradient Boosting':   '#c17c32',
}

CMAP_ROC = LinearSegmentedColormap.from_list(
    'roc_seq', ['#f7fbff', '#deebf7', '#9ecae1', '#4292c6', '#08519c']
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = '../../data/RT_Thorax_v1.csv'
FIG_PATH  = '../figures/'
os.makedirs(FIG_PATH, exist_ok=True)

In [ ]:
# ── Load data ──────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')

TARGET = 'Pathologie_cardiaque'
EXCLUDE_COLS = ['numcent', 'deces', 'time', 'time_cut', 'age_exit', 'age_exit_cut']

CONTINUOUS_VARS  = ['age_diag', 'do_anthra_1K', 'Year_date_diag', 'mean', 'V5', 'V10', 'V15', 'V20']
BINARY_VARS      = ['chimiotherapie_1K', 'anthra_1K', 'CT_sans_anthra']
CATEGORICAL_VARS = ['ctr', 'Year_date_diag_cut', 'age_diag_cut', 'iccc_type', 'gender']
ALL_CANDIDATE_FEATURES = CONTINUOUS_VARS + BINARY_VARS + CATEGORICAL_VARS

y = df[TARGET].values

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'Prevalence: {y.mean():.1%}  ({y.sum()}/{len(y)})')
df.head(3)

---
## 2. Analyse univariee multi-modeles

Pour chaque feature, on entraine 4 modeles (LR, RF, GBDT, SVM) en 5-fold CV stratifie et on mesure :
- ROC-AUC, Sensitivity, Specificity, Balanced Accuracy
- Tests statistiques : Mann-Whitney U (continues) / Chi-2 (categorielles)
- Mutual Information

In [ ]:
# ── Mutual Information ─────────────────────────────────────────
X_for_mi = df[CONTINUOUS_VARS + BINARY_VARS].copy()
for var in CATEGORICAL_VARS:
    dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
    X_for_mi = pd.concat([X_for_mi, dummies], axis=1)

discrete_mask = (
    [False] * len(CONTINUOUS_VARS)
    + [True] * len(BINARY_VARS)
    + [True] * (X_for_mi.shape[1] - len(CONTINUOUS_VARS) - len(BINARY_VARS))
)
mi_scores = mutual_info_classif(X_for_mi, y, discrete_features=discrete_mask,
                                 random_state=RANDOM_STATE)

# Aggregate MI for categorical variables (mean across dummies)
mi_results = {}
for i, var in enumerate(CONTINUOUS_VARS + BINARY_VARS):
    mi_results[var] = mi_scores[i]

for var in CATEGORICAL_VARS:
    var_cols = [c for c in X_for_mi.columns if c.startswith(f'{var}_')]
    if var_cols:
        start_idx = list(X_for_mi.columns).index(var_cols[0])
        mi_results[var] = np.mean([mi_scores[start_idx + i] for i in range(len(var_cols))])
    else:
        mi_results[var] = 0.0

mi_df = pd.DataFrame(list(mi_results.items()), columns=['feature', 'MI'])
mi_df = mi_df.sort_values('MI', ascending=False)
print('Mutual Information scores:')
mi_df

In [ ]:
# ── Model definitions ──────────────────────────────────────────
def get_univariate_models():
    """Return dict of short-name -> model/pipeline for univariate analysis."""
    return {
        'LogReg': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000)),
        ]),
        'RF': RandomForestClassifier(
            n_estimators=100, max_depth=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
        ),
        'GBDT': GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            random_state=RANDOM_STATE,
        ),
        'SVM': Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(
                class_weight='balanced', random_state=RANDOM_STATE, probability=True)),
        ]),
    }


def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)


scorers = {
    'roc_auc':           'roc_auc',
    'sensitivity':       make_scorer(recall_score),
    'specificity':       make_scorer(specificity_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
}

In [ ]:
def univariate_analysis(df, feature, y, cv, feat_type='continuous'):
    """Run all 4 models on a single feature, return metrics dict per model."""
    if feat_type in ('continuous', 'binary'):
        X = df[feature].values.reshape(-1, 1)
    else:
        X = pd.get_dummies(df[feature], prefix=feature, drop_first=True).values

    models = get_univariate_models()
    results = {}
    for name, model in models.items():
        try:
            cv_res = cross_validate(model, X, y, cv=cv, scoring=scorers, n_jobs=-1)
            results[name] = {
                'roc_auc':      cv_res['test_roc_auc'].mean(),
                'roc_auc_std':  cv_res['test_roc_auc'].std(),
                'sensitivity':  cv_res['test_sensitivity'].mean(),
                'specificity':  cv_res['test_specificity'].mean(),
                'balanced_acc': cv_res['test_balanced_accuracy'].mean(),
            }
        except Exception:
            results[name] = {
                'roc_auc': np.nan, 'roc_auc_std': np.nan,
                'sensitivity': np.nan, 'specificity': np.nan, 'balanced_acc': np.nan,
            }
    return results


# ── Run univariate analysis on all features ──────────────────
rows_uni = []

for var in CONTINUOUS_VARS:
    print(f'  {var} (continue)', end=' ')
    res = univariate_analysis(df, var, y, cv, 'continuous')
    stat, pval = mannwhitneyu(df.loc[y == 1, var], df.loc[y == 0, var], alternative='two-sided')
    for m, metrics in res.items():
        rows_uni.append({'feature': var, 'type': 'continue', 'model': m,
                         'p_value': pval, 'MI': mi_results.get(var, 0), **metrics})
    print('OK')

for var in BINARY_VARS:
    print(f'  {var} (binaire)', end=' ')
    res = univariate_analysis(df, var, y, cv, 'binary')
    chi2, pval, *_ = chi2_contingency(pd.crosstab(df[var], df[TARGET]))
    for m, metrics in res.items():
        rows_uni.append({'feature': var, 'type': 'binaire', 'model': m,
                         'p_value': pval, 'MI': mi_results.get(var, 0), **metrics})
    print('OK')

for var in CATEGORICAL_VARS:
    print(f'  {var} (categorielle)', end=' ')
    res = univariate_analysis(df, var, y, cv, 'categorical')
    chi2, pval, *_ = chi2_contingency(pd.crosstab(df[var], df[TARGET]))
    for m, metrics in res.items():
        rows_uni.append({'feature': var, 'type': 'categorielle', 'model': m,
                         'p_value': pval, 'MI': mi_results.get(var, 0), **metrics})
    print('OK')

df_uni = pd.DataFrame(rows_uni)
print(f'\nAnalyse univariee terminee: {len(df_uni)} lignes')

### 2.1 Heatmap ROC-AUC (feature x modele)

In [ ]:
pivot_auc = df_uni.pivot_table(index='feature', columns='model',
                               values='roc_auc', aggfunc='first')
pivot_auc = pivot_auc.reindex(columns=['LogReg', 'RF', 'GBDT', 'SVM'])
pivot_auc['mean_auc'] = pivot_auc.mean(axis=1)
pivot_auc = pivot_auc.sort_values('mean_auc', ascending=True).drop(columns='mean_auc')

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(pivot_auc, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5,
            vmin=0.45, vmax=0.75, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'ROC-AUC'})
ax.set_title('ROC-AUC univarie par feature et modele (5-fold CV)', fontweight='bold')
ax.set_xlabel('Modele')
ax.set_ylabel('Feature')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}11_univariate_heatmap_models.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

### 2.2 Ranking barplots par modele

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
models_list = ['LogReg', 'RF', 'GBDT', 'SVM']

for idx, model_name in enumerate(models_list):
    ax = axes[idx // 2, idx % 2]
    color = COLORS_MODELS[model_name]

    md = df_uni[df_uni['model'] == model_name].copy().sort_values('roc_auc', ascending=True)
    bar_colors = [color if p < 0.05 else '#b0b8c4' for p in md['p_value']]

    ax.barh(md['feature'], md['roc_auc'], color=bar_colors, edgecolor='black', alpha=0.85)
    ax.axvline(x=0.5, color='gray', ls='--', lw=1.5, label='Baseline 0.5')
    ax.set_xlabel('ROC-AUC')
    ax.set_title(f'{model_name} — Ranking univarie', fontweight='bold')
    ax.set_xlim(0.40, 0.75)

    for i, (feat, auc) in enumerate(zip(md['feature'], md['roc_auc'])):
        ax.text(auc + 0.005, i, f'{auc:.3f}', va='center', fontsize=8)

handles = [
    mpatches.Patch(color=COLORS_MODELS['LogReg'], label='p < 0.05'),
    mpatches.Patch(color='#b0b8c4', label='Non significatif'),
]
fig.legend(handles=handles, loc='lower center', ncol=2, fontsize=11)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(f'{FIG_PATH}12_univariate_ranking_by_model.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

### 2.3 Sensitivity vs Specificity

In [ ]:
df_avg = df_uni.groupby('feature').agg(
    sensitivity=('sensitivity', 'mean'),
    specificity=('specificity', 'mean'),
    roc_auc=('roc_auc', 'mean'),
).reset_index().sort_values('roc_auc', ascending=False)

x = np.arange(len(df_avg))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
ax.bar(x - width / 2, df_avg['sensitivity'], width, label='Sensitivity',
       color='#c17c32', alpha=0.85)
ax.bar(x + width / 2, df_avg['specificity'], width, label='Specificity',
       color='#1f4e79', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_avg['feature'], rotation=45, ha='right')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5)
ax.set_ylabel('Score')
ax.set_title('Sensitivity vs Specificity (moyenne sur 4 modeles)', fontweight='bold')
ax.legend()
fig.tight_layout()
fig.savefig(f'{FIG_PATH}13_sensitivity_specificity_comparison.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

### 2.4 Tableau synthetique (p-value, MI, meilleur AUC)

In [ ]:
summary_rows = []
for feat in ALL_CANDIDATE_FEATURES:
    sub = df_uni[df_uni['feature'] == feat]
    pval = sub['p_value'].iloc[0]
    mi   = sub['MI'].iloc[0]
    best_auc = sub['roc_auc'].max()
    best_model = sub.loc[sub['roc_auc'].idxmax(), 'model']
    ftype = sub['type'].iloc[0]
    summary_rows.append({
        'Feature': feat,
        'Type': ftype,
        'p-value': f'{pval:.2e}' if pval < 0.001 else f'{pval:.4f}',
        'MI': f'{mi:.4f}',
        'Best AUC': f'{best_auc:.3f}',
        'Best Model': best_model,
        'Sig.': 'Yes' if pval < 0.05 else 'No',
    })

df_summary_uni = pd.DataFrame(summary_rows)
df_summary_uni

---
## 3. Protocole de selection des features

Processus en 6 etapes pour arriver a un ensemble final de features :

| Etape | Critere | Details |
|---|---|---|
| 1 | p < 0.05 | Filtre statistique (Mann-Whitney / Chi-2) |
| 2 | AUC > 0.55 en CV | Performance univariee minimale |
| 3 | Redondance r > 0.85 | Garder `mean` plutot que V5-V20 |
| 4 | Validation clinique | Coherence avec la litterature (Bentriou 2024, Chounta 2023) |
| 5 | Nested CV | Validation croisee imbriquee |
| 6 | Etude d'ablation | Confirmer chaque variable |

In [ ]:
# -------------------------------------------------------------------------
# Protocole de selection en 6 etapes.
#   Etapes 1-3 : FILTRE STATISTIQUE (data-driven).
#   Etape 4    : VALIDATION CLINIQUE -> ajoute des confondants etablis par la
#                litterature meme si leur signal univarie en classification est
#                faible (choix d'AJUSTEMENT, pas de performance).
# -------------------------------------------------------------------------

# Etape 1 — Filtre statistique (Mann-Whitney / Chi-2, p < 0.05)
sig_features = df_uni[df_uni['p_value'] < 0.05]['feature'].unique().tolist()
print(f'Etape 1 — significatives p<0.05 [{len(sig_features)}]: {sorted(sig_features)}')

# Etape 2 — AUC univarie > 0.55 (meilleur modele) ET significatif
best_auc_per_feat = df_uni.groupby('feature')['roc_auc'].max()
auc_features = best_auc_per_feat[best_auc_per_feat > 0.55].index.tolist()
step2 = sorted(set(sig_features) & set(auc_features))
print(f'Etape 2 — significatif & AUC>0.55 [{len(step2)}]: {step2}')

# Etape 3 — Suppression des redondances
#   (a) V5-V20 correlees a `mean` (r>0.85) -> on garde `mean`
#   (b) doublons categoriels d'une continue deja presente (ex. Year_date_diag_cut)
dose_vars = ['mean', 'V5', 'V10', 'V15', 'V20']
corr_dose = df[dose_vars].corr()
print('\nCorrelation dosimetrique (mean vs Vx):')
print(corr_dose.loc['mean', ['V5', 'V10', 'V15', 'V20']].round(3).to_string())
redundant_dose = [v for v in ['V5', 'V10', 'V15', 'V20'] if v in step2]
redundant_cut  = [v for v in ['Year_date_diag_cut', 'age_diag_cut'] if v in step2]
step3 = [f for f in step2 if f not in redundant_dose + redundant_cut]
print(f'\nEtape 3 — apres redondances [{len(step3)}]: {step3}')
print(f'   retires: {redundant_dose + redundant_cut}')

# Etape 4 — Validation clinique (client + litterature Bentriou 2024 / Chounta 2023)
#   - retrait de `ctr` (centre de traitement : variable administrative, non causale)
#   - `do_anthra_1K` (continue, tres asymetrique) remplacee par sa forme binaire `anthra_1K`
#   - AJOUT de confondants cliniques etablis, faibles en univarie de classification mais
#     incontournables : `age_diag` (age a l'irradiation) et `anthra_1K` (cardiotoxicite
#     dose-dependante ; significatif en survie : HR~1.7, log-rank p~1e-4)
stat_kept = [f for f in step3 if f not in ('ctr', 'do_anthra_1K')]
FORCED_CLINICAL = ['age_diag', 'anthra_1K']
print(f'\nEtape 4 — retenu apres validation clinique (hors ctr/do_anthra_1K): {stat_kept}')
print(f'   + confondants cliniques imposes: {FORCED_CLINICAL}')

# Ensemble final (dose, ere, age, type de cancer, anthracyclines)
SELECTED_FEATURES = ['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']
assert set(stat_kept).issubset(set(SELECTED_FEATURES)), (stat_kept, SELECTED_FEATURES)
print(f'\n>>> Features finales [{len(SELECTED_FEATURES)}]: {SELECTED_FEATURES}')
print('    (etapes 5-6 : nested CV + ablations ci-dessous confirment cet ensemble)')

---
## 4. Etude d'ablation — gender

Gender a un AUC univarie de ~0.51 et un p-value non significatif (Chi-2).
On compare les performances avec et sans gender pour confirmer son exclusion.

In [ ]:
def build_pipeline(features, model_key):
    """Build preprocessing + model pipeline."""
    continuous  = [f for f in features if f in CONTINUOUS_VARS]
    categorical = [f for f in features if f in CATEGORICAL_VARS]
    binary      = [f for f in features if f in BINARY_VARS]

    transformers = []
    if continuous:
        transformers.append(('num', StandardScaler(), continuous))
    if categorical:
        transformers.append(('cat', OneHotEncoder(drop='first', sparse_output=False,
                                                   handle_unknown='ignore'), categorical))
    if binary:
        transformers.append(('bin', 'passthrough', binary))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    estimators = {
        'Logistic Regression': LogisticRegression(
            C=1, solver='lbfgs', penalty='l2',
            class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=5, min_samples_split=5, min_samples_leaf=10,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            min_samples_split=10, min_samples_leaf=5, random_state=RANDOM_STATE),
    }

    return Pipeline([('preprocessor', preprocessor), ('classifier', estimators[model_key])])


def fit_balanced(pipe, X_tr, y_tr, model_key):
    """Fit pipeline avec ponderation equilibree des classes.
    LR et RF utilisent class_weight='balanced'; GradientBoosting n'a pas ce parametre,
    on l'equilibre donc via sample_weight (meme effet)."""
    if model_key == 'Gradient Boosting':
        sw = compute_sample_weight('balanced', y_tr)
        pipe.fit(X_tr, y_tr, classifier__sample_weight=sw)
    else:
        pipe.fit(X_tr, y_tr)
    return pipe


def evaluate_config(df, features, y, config_name, cv=cv):
    """Evaluate 3 models on a feature set; return per-model curves + metrics."""
    X = df[features].copy()
    results = {}

    for model_key in MODEL_COLORS_LONG:
        fold_metrics = []
        all_fpr = np.linspace(0, 1, 100)
        all_tpr = []
        all_recall_grid = np.linspace(0, 1, 100)
        all_precision = []

        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model = fit_balanced(build_pipeline(features, model_key), X_train, y_train, model_key)
            y_proba = model.predict_proba(X_test)[:, 1]

            fpr, tpr, _ = roc_curve(y_test, y_proba)
            all_tpr.append(np.interp(all_fpr, fpr, tpr))

            prec, rec, _ = precision_recall_curve(y_test, y_proba)
            all_precision.append(np.interp(all_recall_grid, rec[::-1], prec[::-1]))

            fold_metrics.append({
                'roc_auc': roc_auc_score(y_test, y_proba),
                'pr_auc':  average_precision_score(y_test, y_proba),
            })

        fm = pd.DataFrame(fold_metrics)
        results[model_key] = {
            'roc_auc_mean':  fm['roc_auc'].mean(),
            'roc_auc_std':   fm['roc_auc'].std(),
            'pr_auc_mean':   fm['pr_auc'].mean(),
            'pr_auc_std':    fm['pr_auc'].std(),
            'mean_tpr':      np.mean(all_tpr, axis=0),
            'std_tpr':       np.std(all_tpr, axis=0),
            'mean_prec':     np.mean(all_precision, axis=0),
            'std_prec':      np.std(all_precision, axis=0),
            'all_fpr':       all_fpr,
            'all_recall_grid': all_recall_grid,
        }

    return results

In [ ]:
feats_with_gender = ['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K', 'gender']
feats_no_gender   = ['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']

res_with = evaluate_config(df, feats_with_gender, y, 'avec gender')
res_without = evaluate_config(df, feats_no_gender, y, 'sans gender')

In [ ]:
def plot_roc_pr_comparison(results_dict, title_suffix, filename):
    """ROC + PR side by side for multiple configs."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1))]

    for cfg_idx, (config_name, results) in enumerate(results_dict.items()):
        ls = linestyles[cfg_idx % len(linestyles)]
        for model_key, res in results.items():
            color = MODEL_COLORS_LONG[model_key]
            label = (f'{model_key} -- {config_name}\n'
                     f'(AUC={res["roc_auc_mean"]:.3f} +/- {res["roc_auc_std"]:.3f})')

            axes[0].plot(res['all_fpr'], res['mean_tpr'], color=color, lw=2,
                         linestyle=ls, label=label)
            if cfg_idx == 0:
                axes[0].fill_between(
                    res['all_fpr'],
                    res['mean_tpr'] - res['std_tpr'],
                    res['mean_tpr'] + res['std_tpr'],
                    alpha=0.1, color=color)

            label_pr = (f'{model_key} -- {config_name}\n'
                        f'(AUC={res["pr_auc_mean"]:.3f} +/- {res["pr_auc_std"]:.3f})')
            axes[1].plot(res['all_recall_grid'], res['mean_prec'], color=color, lw=2,
                         linestyle=ls, label=label_pr)

    axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
    axes[0].set_title(f'Courbes ROC -- {title_suffix}', fontweight='bold')
    axes[0].legend(loc='lower right', fontsize=7)

    prev = y.mean()
    axes[1].axhline(prev, color='gray', ls=':', lw=1, alpha=0.6, label=f'Prevalence ({prev:.2f})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title(f'Courbes PR -- {title_suffix}', fontweight='bold')
    axes[1].legend(loc='upper right', fontsize=7)

    fig.tight_layout()
    fig.savefig(f'{FIG_PATH}{filename}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


plot_roc_pr_comparison(
    {'Avec gender': res_with, 'Sans gender': res_without},
    'Ablation de gender',
    'ablation_gender_roc_pr',
)

In [ ]:
# Delta table
print(f'{"Modele":<25} {"DROC-AUC":>12} {"DPR-AUC":>12}')
print('-' * 50)
gender_deltas = []
for model_key in MODEL_COLORS_LONG:
    d_roc = res_without[model_key]['roc_auc_mean'] - res_with[model_key]['roc_auc_mean']
    d_pr  = res_without[model_key]['pr_auc_mean']  - res_with[model_key]['pr_auc_mean']
    print(f'{model_key:<25} {d_roc:+.4f}       {d_pr:+.4f}')
    gender_deltas.append({'Model': model_key, 'delta_ROC': d_roc, 'delta_PR': d_pr})

pd.DataFrame(gender_deltas).to_csv(f'{FIG_PATH}ablation_gender_results.csv', index=False)
print('\nConclusion: gender n\'apporte pas de gain (AUC univarie ~0.51, Cox HR=1.02, p=0.88). Exclu.')

---
## 5. Etude d'ablation — variables dosimetriques

Comparaison de `mean` vs `V5`, `V10`, `V15`, `V20` comme variable dosimetrique dans le modele multivariate.
Les variables Vx sont fortement correlees avec `mean` (r > 0.85).

In [ ]:
base_non_dose = ['Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']  # = ensemble final hors dose
dose_configs = {
    'mean': ['mean'] + base_non_dose,
    'V5':   ['V5']   + base_non_dose,
    'V10':  ['V10']  + base_non_dose,
    'V15':  ['V15']  + base_non_dose,
    'V20':  ['V20']  + base_non_dose,
}

dose_results = {}
for dose_name, features in dose_configs.items():
    print(f'  Evaluating {dose_name}...')
    dose_results[dose_name] = evaluate_config(df, features, y, dose_name)

print('Done.')

In [ ]:
# Build summary table
dose_rows = []
for dose_name, results in dose_results.items():
    for model_key, res in results.items():
        dose_rows.append({
            'Dose_var': dose_name, 'Model': model_key,
            'ROC_AUC_mean': res['roc_auc_mean'], 'ROC_AUC_std': res['roc_auc_std'],
            'PR_AUC_mean':  res['pr_auc_mean'],  'PR_AUC_std':  res['pr_auc_std'],
        })
dose_df = pd.DataFrame(dose_rows)

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_roc = dose_df.pivot(index='Model', columns='Dose_var', values='ROC_AUC_mean')
pivot_roc = pivot_roc[['mean', 'V5', 'V10', 'V15', 'V20']]
sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0],
            vmin=pivot_roc.values.min() - 0.01, vmax=pivot_roc.values.max() + 0.01)
axes[0].set_title('ROC-AUC par variable dosimetrique', fontweight='bold')
axes[0].set_ylabel('')

pivot_pr = dose_df.pivot(index='Model', columns='Dose_var', values='PR_AUC_mean')
pivot_pr = pivot_pr[['mean', 'V5', 'V10', 'V15', 'V20']]
sns.heatmap(pivot_pr, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1],
            vmin=pivot_pr.values.min() - 0.01, vmax=pivot_pr.values.max() + 0.01)
axes[1].set_title('PR-AUC par variable dosimetrique', fontweight='bold')
axes[1].set_ylabel('')

fig.suptitle('Ablation dosimetrique : mean vs V5/V10/V15/V20',
             fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_PATH}ablation_dose_heatmap.png', dpi=300,
            bbox_inches='tight', facecolor='white')
dose_df.to_csv(f'{FIG_PATH}ablation_dose_results.csv', index=False)
plt.show()
print('Conclusion: mean domine ou egalise les Vx. On retient mean (plus interpretable + meilleure couverture).')

---
## 6. Modeles multivaries (5-fold CV)

Feature set final : `['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']`

Modeles (ponderation des classes equilibree pour les trois) :
- **Logistic Regression** : C=1, `class_weight='balanced'`
- **Random Forest** : n_estimators=200, max_depth=5, `class_weight='balanced'`
- **Gradient Boosting** : n_estimators=100, max_depth=4, `sample_weight` equilibre

Preprocessing : StandardScaler (continues) + OneHotEncoder (categorielles)

In [ ]:
FINAL_FEATURES = ['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']

X_final = df[FINAL_FEATURES].copy()
multivar_results = {}

for model_key in MODEL_COLORS_LONG:
    fold_metrics = []
    all_fpr = np.linspace(0, 1, 100)
    all_tpr = []
    all_recall_grid = np.linspace(0, 1, 100)
    all_precision = []

    for train_idx, test_idx in cv.split(X_final, y):
        X_train, X_test = X_final.iloc[train_idx], X_final.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipe = fit_balanced(build_pipeline(FINAL_FEATURES, model_key), X_train, y_train, model_key)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        all_tpr.append(np.interp(all_fpr, fpr, tpr))

        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        all_precision.append(np.interp(all_recall_grid, rec[::-1], prec[::-1]))

        fold_metrics.append({
            'roc_auc': roc_auc_score(y_test, y_proba),
            'pr_auc':  average_precision_score(y_test, y_proba),
            'f1':      f1_score(y_test, y_pred, zero_division=0),
            'recall':  recall_score(y_test, y_pred, zero_division=0),
            'bal_acc': balanced_accuracy_score(y_test, y_pred),
        })

    fm = pd.DataFrame(fold_metrics)
    multivar_results[model_key] = {
        'roc_auc_mean': fm['roc_auc'].mean(), 'roc_auc_std': fm['roc_auc'].std(),
        'pr_auc_mean':  fm['pr_auc'].mean(),  'pr_auc_std':  fm['pr_auc'].std(),
        'f1_mean':      fm['f1'].mean(),       'f1_std':      fm['f1'].std(),
        'recall_mean':  fm['recall'].mean(),   'recall_std':  fm['recall'].std(),
        'bal_acc_mean': fm['bal_acc'].mean(),  'bal_acc_std': fm['bal_acc'].std(),
        'mean_tpr':     np.mean(all_tpr, axis=0),
        'std_tpr':      np.std(all_tpr, axis=0),
        'mean_prec':    np.mean(all_precision, axis=0),
        'std_prec':     np.std(all_precision, axis=0),
        'all_fpr':      all_fpr,
        'all_recall_grid': all_recall_grid,
    }

print('Multivariate evaluation done.')

### 6.1 Courbes ROC et PR (avec bandes +/- std)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

for model_key, res in multivar_results.items():
    color = MODEL_COLORS_LONG[model_key]
    fpr = res['all_fpr']

    # ROC
    label = f'{model_key}\n(AUC={res["roc_auc_mean"]:.3f} +/- {res["roc_auc_std"]:.3f})'
    axes[0].plot(fpr, res['mean_tpr'], color=color, lw=2, label=label)
    axes[0].fill_between(fpr,
                         res['mean_tpr'] - res['std_tpr'],
                         res['mean_tpr'] + res['std_tpr'],
                         alpha=0.12, color=color)

    # PR
    label_pr = f'{model_key}\n(AUC={res["pr_auc_mean"]:.3f} +/- {res["pr_auc_std"]:.3f})'
    axes[1].plot(res['all_recall_grid'], res['mean_prec'], color=color, lw=2, label=label_pr)
    axes[1].fill_between(res['all_recall_grid'],
                         res['mean_prec'] - res['std_prec'],
                         res['mean_prec'] + res['std_prec'],
                         alpha=0.12, color=color)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Courbes ROC -- Modeles multivaries (5-fold CV)', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)

prev = y.mean()
axes[1].axhline(prev, color='gray', ls=':', lw=1, alpha=0.6,
                label=f'Prevalence ({prev:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Courbes PR -- Modeles multivaries (5-fold CV)', fontweight='bold')
axes[1].legend(loc='upper right', fontsize=9)

fig.tight_layout()
fig.savefig(f'{FIG_PATH}roc_pr_with_uncertainty.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### 6.2 Tableau recapitulatif

In [ ]:
rows_mv = []
for model_key, res in multivar_results.items():
    rows_mv.append({
        'Model': model_key,
        'ROC-AUC':      f'{res["roc_auc_mean"]:.3f} +/- {res["roc_auc_std"]:.3f}',
        'PR-AUC':       f'{res["pr_auc_mean"]:.3f} +/- {res["pr_auc_std"]:.3f}',
        'F1':           f'{res["f1_mean"]:.3f} +/- {res["f1_std"]:.3f}',
        'Recall':       f'{res["recall_mean"]:.3f} +/- {res["recall_std"]:.3f}',
        'Balanced Acc': f'{res["bal_acc_mean"]:.3f} +/- {res["bal_acc_std"]:.3f}',
    })

df_multivar_summary = pd.DataFrame(rows_mv)
df_multivar_summary

### 6.3 Importance des variables et coefficients

Importance des variables (Random Forest, Gini) et coefficients (Regression Logistique,
entrees standardisees) sur l'ensemble final, reentraine sur toute la cohorte.
Les modalites one-hot d'une variable categorielle sont agregees par variable.
Figures `15_feature_importance.png` et `16_lr_coefficients.png` (utilisees dans les slides).

In [ ]:
def _parent_feature(name):
    """Map an encoded column (e.g. 'iccc_type_06 -Renal tumors') back to its source feature."""
    for f in FINAL_FEATURES:
        if name == f or name.startswith(f + '_'):
            return f
    return name

# ---- Random Forest : importances (Gini) agregees par variable ----
rf_pipe = fit_balanced(build_pipeline(FINAL_FEATURES, 'Random Forest'), X_final, y, 'Random Forest')
enc_names = [n.split('__', 1)[1] if '__' in n else n
            for n in rf_pipe.named_steps['preprocessor'].get_feature_names_out()]
rf_imp = defaultdict(float)
for n, v in zip(enc_names, rf_pipe.named_steps['classifier'].feature_importances_):
    rf_imp[_parent_feature(n)] += v
rf_series = pd.Series(rf_imp).reindex(FINAL_FEATURES).sort_values()

# ---- Logistic Regression : coefficient le plus marquant par variable ----
lr_pipe = fit_balanced(build_pipeline(FINAL_FEATURES, 'Logistic Regression'), X_final, y, 'Logistic Regression')
lr_coef = defaultdict(float)
for n, c in zip(enc_names, lr_pipe.named_steps['classifier'].coef_[0]):
    p = _parent_feature(n)
    if abs(c) > abs(lr_coef[p]):
        lr_coef[p] = c
lr_series = pd.Series(lr_coef).reindex(FINAL_FEATURES).sort_values()

# ---- Figure 15 : RF importances ----
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(rf_series.index, rf_series.values, color=MODEL_COLORS_LONG['Random Forest'],
        edgecolor='black', alpha=0.85)
for i, v in enumerate(rf_series.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Importance (Gini, agregee par variable)')
ax.set_title('Importance des variables — Random Forest', fontweight='bold')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}15_feature_importance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# ---- Figure 16 : LR coefficients ----
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#c0392b' if v > 0 else '#1f4e79' for v in lr_series.values]
ax.barh(lr_series.index, lr_series.values, color=bar_colors, edgecolor='black', alpha=0.85)
ax.axvline(0, color='gray', lw=1)
for i, v in enumerate(lr_series.values):
    ax.text(v + (0.02 if v >= 0 else -0.02), i, f'{v:+.2f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlabel('Coefficient (entrees standardisees) — positif = augmente le risque')
ax.set_title('Coefficients — Regression Logistique', fontweight='bold')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}16_lr_coefficients.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Importance RF (desc):'); print(rf_series.sort_values(ascending=False).round(4).to_string())
print('\nLR coef |max| par variable:'); print(lr_series.round(3).to_string())
print('\n-> mean domine dans les deux modeles (coherence inter-modeles).')

---
## 7. Nested CV feature selection (model-specific)

Protocole :
- **Outer** : 5-fold stratifie
- **Inner** : 3-fold stratifie (selection sur train seulement)
- Top-K = 6 features
- Pour chaque modele, on selectionne les features optimales sur les donnees d'entrainement uniquement (pas de fuite d'information)
- Comparaison : common feature set vs model-specific

In [ ]:
OUTER_SPLITS = 5
INNER_SPLITS = 3
TOP_K = 6

COMMON_FEATURE_SET = ['mean', 'Year_date_diag', 'age_diag', 'iccc_type', 'anthra_1K']  # = ensemble final (gender exclu)
NESTED_MODEL_KEYS = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']


def feature_type(feature):
    if feature in CONTINUOUS_VARS:
        return 'continuous'
    if feature in BINARY_VARS:
        return 'binary'
    return 'categorical'


def get_estimator(model_key):
    if model_key == 'Logistic Regression':
        return LogisticRegression(
            class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000, solver='lbfgs')
    if model_key == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_split=10, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    if model_key == 'Gradient Boosting':
        return GradientBoostingClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            min_samples_split=10, min_samples_leaf=5, random_state=RANDOM_STATE)
    raise ValueError(f'Unknown model: {model_key}')


def make_nested_pipeline(model_key, selected_features):
    continuous  = [f for f in selected_features if f in CONTINUOUS_VARS]
    categorical = [f for f in selected_features if f in CATEGORICAL_VARS]
    binary      = [f for f in selected_features if f in BINARY_VARS]
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical),
            ('bin', 'passthrough', binary),
        ],
        remainder='drop',
    )
    return Pipeline([('preprocessor', preprocessor), ('classifier', get_estimator(model_key))])


def make_univariate_matrix(df_sub, feature):
    ft = feature_type(feature)
    if ft in ('continuous', 'binary'):
        return df_sub[[feature]].values
    encoded = pd.get_dummies(df_sub[feature], prefix=feature, drop_first=True)
    return encoded.values if encoded.shape[1] > 0 else np.zeros((len(df_sub), 1))


def univariate_auc_inner(df_train, y_train, feature, model_key, inner_cv):
    X_feat = make_univariate_matrix(df_train, feature)
    aucs = []
    for itr, iva in inner_cv.split(X_feat, y_train):
        X_tr, X_va = X_feat[itr], X_feat[iva]
        y_tr, y_va = y_train[itr], y_train[iva]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            continue
        if model_key == 'Logistic Regression':
            m = Pipeline([('scaler', StandardScaler()), ('clf', get_estimator(model_key))])
        else:
            m = get_estimator(model_key)
        m.fit(X_tr, y_tr)
        y_proba = m.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, y_proba))
    return float(np.mean(aucs)) if aucs else 0.5


def feature_pvalue(df_train, y_train, feature):
    tmp = df_train.copy()
    tmp[TARGET] = y_train
    ft = feature_type(feature)
    if ft == 'continuous':
        pos = tmp.loc[tmp[TARGET] == 1, feature]
        neg = tmp.loc[tmp[TARGET] == 0, feature]
        if len(pos) == 0 or len(neg) == 0:
            return 1.0
        _, p = mannwhitneyu(pos, neg, alternative='two-sided')
        return float(p)
    ct = pd.crosstab(tmp[feature], tmp[TARGET])
    if ct.shape[0] <= 1 or ct.shape[1] <= 1:
        return 1.0
    _, p, _, _ = chi2_contingency(ct)
    return float(p)


def select_features_for_model(df_train, y_train, model_key, top_k=TOP_K):
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for feat in ALL_CANDIDATE_FEATURES:
        pval = feature_pvalue(df_train, y_train, feat)
        auc  = univariate_auc_inner(df_train, y_train, feat, model_key, inner_cv)
        rows.append({'feature': feat, 'p_value': pval, 'inner_auc': auc})
    scores = pd.DataFrame(rows).sort_values('inner_auc', ascending=False)
    selected = scores[(scores['p_value'] < 0.05) & (scores['inner_auc'] > 0.5)]['feature'].tolist()
    if len(selected) < 2:
        selected = scores.head(min(top_k, len(scores)))['feature'].tolist()
    else:
        selected = selected[:top_k]
    return selected, scores


def evaluate_on_fold(model_key, selected_features, X_train_df, y_train, X_test_df, y_test):
    pipe = fit_balanced(make_nested_pipeline(model_key, selected_features),
                        X_train_df[selected_features], y_train, model_key)
    y_proba = pipe.predict_proba(X_test_df[selected_features])[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    return {
        'roc_auc':  roc_auc_score(y_test, y_proba),
        'pr_auc':   average_precision_score(y_test, y_proba),
        'f1':       f1_score(y_test, y_pred, zero_division=0),
        'recall':   recall_score(y_test, y_pred, zero_division=0),
        'bal_acc':  balanced_accuracy_score(y_test, y_pred),
    }

In [ ]:
outer_cv = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)

baseline_metrics = defaultdict(list)
level2_metrics   = defaultdict(list)
feature_freq     = {m: Counter() for m in NESTED_MODEL_KEYS}
selected_by_fold = []

print('Nested CV en cours...')
for fold_idx, (tr_idx, te_idx) in enumerate(outer_cv.split(df, y), start=1):
    print(f'  Outer fold {fold_idx}/{OUTER_SPLITS}')
    X_tr, X_te = df.iloc[tr_idx], df.iloc[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    for mk in NESTED_MODEL_KEYS:
        # Baseline
        bm = evaluate_on_fold(mk, COMMON_FEATURE_SET, X_tr, y_tr, X_te, y_te)
        bm['n_features'] = len(COMMON_FEATURE_SET)
        baseline_metrics[mk].append(bm)

        # Model-specific
        selected, _ = select_features_for_model(X_tr, y_tr, mk, top_k=TOP_K)
        for feat in selected:
            feature_freq[mk][feat] += 1

        selected_by_fold.append({
            'fold': fold_idx, 'model': mk,
            'selected': ', '.join(selected), 'n': len(selected),
        })

        l2m = evaluate_on_fold(mk, selected, X_tr, y_tr, X_te, y_te)
        l2m['n_features'] = len(selected)
        level2_metrics[mk].append(l2m)

print('Done.')

### 7.1 Comparaison : common set vs model-specific

In [ ]:
def agg_metrics(rows, model_key, setup):
    d = pd.DataFrame(rows)
    return {
        'Model': model_key, 'Setup': setup,
        'ROC-AUC': f'{d["roc_auc"].mean():.3f} +/- {d["roc_auc"].std():.3f}',
        'PR-AUC':  f'{d["pr_auc"].mean():.3f} +/- {d["pr_auc"].std():.3f}',
        'F1':      f'{d["f1"].mean():.3f} +/- {d["f1"].std():.3f}',
        'Recall':  f'{d["recall"].mean():.3f} +/- {d["recall"].std():.3f}',
        'Balanced Acc': f'{d["bal_acc"].mean():.3f} +/- {d["bal_acc"].std():.3f}',
        'roc_auc_mean': d['roc_auc'].mean(),
        'roc_auc_std':  d['roc_auc'].std(),
        'pr_auc_mean':  d['pr_auc'].mean(),
        'pr_auc_std':   d['pr_auc'].std(),
    }

comp_rows = []
for mk in NESTED_MODEL_KEYS:
    comp_rows.append(agg_metrics(baseline_metrics[mk], mk, 'Common feature set'))
    comp_rows.append(agg_metrics(level2_metrics[mk],   mk, 'Model-specific (nested CV)'))

df_comp = pd.DataFrame(comp_rows)
df_comp[['Model', 'Setup', 'ROC-AUC', 'PR-AUC', 'F1', 'Recall', 'Balanced Acc']]

In [ ]:
# Barplot comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
setup_order = ['Common feature set', 'Model-specific (nested CV)']

sns.barplot(data=df_comp, x='Model', y='roc_auc_mean', hue='Setup',
            hue_order=setup_order, ax=axes[0],
            palette={'Common feature set': '#8da0cb',
                     'Model-specific (nested CV)': '#1f4e79'})
axes[0].set_title('Nested CV ROC-AUC', fontweight='bold')
axes[0].set_xlabel(''); axes[0].set_ylabel('ROC-AUC')
axes[0].set_ylim(0.55, 0.82)
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend(title='Setup', loc='upper left', fontsize=9)

sns.barplot(data=df_comp, x='Model', y='pr_auc_mean', hue='Setup',
            hue_order=setup_order, ax=axes[1],
            palette={'Common feature set': '#b2df8a',
                     'Model-specific (nested CV)': '#2e8b57'})
axes[1].set_title('Nested CV PR-AUC', fontweight='bold')
axes[1].set_xlabel(''); axes[1].set_ylabel('PR-AUC')
axes[1].set_ylim(0.15, 0.50)
axes[1].grid(axis='y', alpha=0.25)
axes[1].legend(title='Setup', loc='upper left', fontsize=9)

fig.tight_layout()
fig.savefig(f'{FIG_PATH}29_nestedcv_model_specific_comparison.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### 7.2 Feature frequency heatmap

In [ ]:
freq_rows = []
for mk, counter in feature_freq.items():
    for feat in ALL_CANDIDATE_FEATURES:
        freq_rows.append({
            'model': mk, 'feature': feat,
            'count': counter.get(feat, 0),
            'rate': counter.get(feat, 0) / OUTER_SPLITS,
        })
df_freq = pd.DataFrame(freq_rows)

freq_pivot = df_freq.pivot_table(
    index='feature', columns='model', values='rate', aggfunc='first'
).reindex(index=ALL_CANDIDATE_FEATURES, columns=NESTED_MODEL_KEYS)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(freq_pivot, annot=True, fmt='.2f', cmap='Blues',
            vmin=0.0, vmax=1.0, linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Selection rate (outer folds)'}, ax=ax)
ax.set_title('Model-specific feature selection frequency (nested CV)', fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
fig.tight_layout()
fig.savefig(f'{FIG_PATH}30_model_specific_feature_frequency_heatmap.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Features selected per fold
df_sel = pd.DataFrame(selected_by_fold)
df_sel

---
## 8. Resume et limites

### Resultats principaux

| Resultat | Valeur |
|---|---|
| Meilleur modele | Random Forest |
| ROC-AUC (5-fold CV) | 0.77 +/- 0.05 |
| PR-AUC | ~0.41 (baseline aleatoire 0.17) |
| Features finales | `mean`, `Year_date_diag`, `age_diag`, `iccc_type`, `anthra_1K` |

### Decisions justifiees
- **gender exclu** : AUC univarie ~0.48 (< 0.50), Chi-2 p~0.50 ; ablation sans gain (Cox HR=1.02, p=0.88).
- **mean retenu** : domine ou egalise V5-V20 dans tous les modeles, plus interpretable cliniquement.
- **age_diag / anthra_1K** : confondants cliniques imposes (litterature + analyse de survie) malgre un signal univarie faible en classification — role d'ajustement, pas de performance (cf. section 6.3).
- **Ponderation equilibree (3 modeles)** : `class_weight='balanced'` (LR, RF) et `sample_weight` equilibre (Gradient Boosting) pour le desequilibre 17.2%.

### Limites
- Plateau de performance ~0.67 (univarie) -> ~0.77 (multivarie) avec les variables cliniques seules.
- Le desequilibre des classes (17.2%) limite la precision (PR-AUC).
- Variables dosimetriques resumees par des scalaires (mean, Vx) — perte de l'information spatiale.

### Perspectives
- Integration des **matrices de dose 3D** voxelisees (`data/coeur_1k/`).
- Deep learning (CNN 3D) + cartes d'activation (Grad-CAM 3D).
- Modeles de survie (Cox, RSF) exploitant `time` et `deces` (cf. notebook 03).

In [ ]:
# Save all CSVs
df_uni.to_csv(f'{FIG_PATH}24_univariate_results.csv', index=False)
df_summary_uni.to_csv(f'{FIG_PATH}23_summary_table.csv', index=False)
df_comp[['Model', 'Setup', 'ROC-AUC', 'PR-AUC', 'F1', 'Recall', 'Balanced Acc']].to_csv(
    f'{FIG_PATH}29_nestedcv_model_specific_scores.csv', index=False)
df_freq.to_csv(f'{FIG_PATH}30_model_specific_feature_frequency.csv', index=False)
df_sel.to_csv(f'{FIG_PATH}31_model_specific_selected_by_fold.csv', index=False)

print('All outputs saved to', FIG_PATH)
print('Notebook 02_Classification complete.')